In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from collections import Counter
from fractions import Fraction
import re

In [ ]:
BASE = r'D:\Research\Learning_CFT\ADE-classification\cft_library\label'

# CSV paths for each central-charge range
PATHS = {
    r'$c<50$':              rf'{BASE}\lower_cc\cft_KM_1M_c50_l2-5_rand_select.csv',
    r'$50\leq c<75$':       rf'{BASE}\rand_select_all_data\cft_KM_c50-75_l2-5_rand_select.csv',
    r'$75\leq c\leq100$':   rf'{BASE}\rand_select_all_data\cft_KM_c75-100_l2-5_rand_select.csv',
    r'$50\leq c\leq100$':   rf'{BASE}\rand_select_all_data\cft_KM_c50-100_l2-5_rand_select.csv',
}

ALGEBRA_ORDER = [
    'su2', 'su3', 'sp4', 'g2',
    'su4', 'so7', 'sp6',
    'su5', 'so8', 'so9', 'sp8', 'f4',
    'su6', 'so10', 'so11', 'so12', 'e6', 'e7', 'e8',
]

BASE_KEY   = r'$c<50$'
REL_KEYS   = [r'$50\leq c<75$', r'$75\leq c\leq100$', r'$50\leq c\leq100$']
REL_COLORS = ['#ff7f0e', '#2ca02c', '#d62728']

LABEL_RE = re.compile(r'\b[a-zA-Z]{1,3}\d+\b')
CC_RE    = re.compile(r'\b[a-zA-Z]{1,3}\d+,(\d+(?:/\d+)?)\b')

In [ ]:
def load_label_pct(csv_path):
    df = pd.read_csv(csv_path, header=None)
    counts = Counter()
    for col in df.columns:
        for matches in df[col].astype(str).str.findall(LABEL_RE):
            counts.update(matches)
    total = sum(counts.get(lbl, 0) for lbl in ALGEBRA_ORDER)
    return {lbl: counts.get(lbl, 0) / total * 100 for lbl in ALGEBRA_ORDER}


def load_cc_denom_pct(csv_path):
    df = pd.read_csv(csv_path, header=None)
    denoms = []
    for col in df.columns:
        for cell in df[col].astype(str):
            for m in CC_RE.findall(cell):
                denoms.append(Fraction(m).denominator)
    counts = Counter(denoms)
    total  = sum(counts.values())
    return {d: counts[d] / total * 100 for d in sorted(counts)}


label_pcts = {name: load_label_pct(path)    for name, path in PATHS.items()}
cc_pcts    = {name: load_cc_denom_pct(path) for name, path in PATHS.items()}

In [ ]:
# Fig 2 LEFT — algebra label frequency in the default training dataset (c < 50)
pct = [label_pcts[BASE_KEY][lbl] for lbl in ALGEBRA_ORDER]

fig, ax = plt.subplots(figsize=(7, 6))
ax.bar(range(len(ALGEBRA_ORDER)), pct)
ax.set_xticks(range(len(ALGEBRA_ORDER)))
ax.set_xticklabels(ALGEBRA_ORDER, rotation=45, fontsize=15)
ax.tick_params(axis='y', labelsize=15)
ax.set_ylabel('Frequency of Algebra Labels (%)', fontsize=20)
ax.yaxis.set_major_locator(MultipleLocator(5))
ax.yaxis.set_minor_locator(MultipleLocator(1))
ax.set_yscale('log', base=10)
plt.tight_layout()
plt.savefig('freq_c50_label_log.pdf')
plt.show()

In [ ]:
# Fig 2 BOTTOM — central charge denominator frequency in the default training dataset (c < 50)
base_cc   = cc_pcts[BASE_KEY]
denoms    = list(base_cc.keys())
denom_pct = list(base_cc.values())

fig, ax = plt.subplots(figsize=(7, 6))
ax.bar(range(len(denoms)), denom_pct, color='#1f77b4')
ax.set_xticks(range(len(denoms)))
ax.set_xticklabels([str(d) for d in denoms], rotation=0, fontsize=17)
ax.set_xlabel('Denominator of Central Charge', fontsize=20)
ax.set_ylabel('Frequency of Denominators (%)', fontsize=20)
ax.tick_params(axis='both', labelsize=17)
ax.set_yscale('log', base=10)
plt.tight_layout()
plt.savefig('freq_c50_cc_log.pdf')
plt.show()

In [ ]:
# Relative frequencies normalised to the c<50 baseline
label_rel = {
    key: [label_pcts[key].get(lbl, 0) / label_pcts[BASE_KEY].get(lbl, 1)
          for lbl in ALGEBRA_ORDER]
    for key in REL_KEYS
}

all_denoms = sorted({d for pct in cc_pcts.values() for d in pct})
cc_rel = {
    key: [cc_pcts[key].get(d, 0) / cc_pcts[BASE_KEY].get(d, 1)
          for d in all_denoms]
    for key in REL_KEYS
}

In [ ]:
# Fig 4 LEFT — relative algebra label frequency (horizontal)
df_label_rel = pd.DataFrame(label_rel, index=ALGEBRA_ORDER)

n_groups  = len(ALGEBRA_ORDER)
n_bars    = len(REL_KEYS)
bar_width = 0.3
group_gap = 1.2

ax = df_label_rel.plot(kind='barh', figsize=(6, 8), width=0.6, color=REL_COLORS)

for i, patch_group in enumerate(zip(*[ax.containers[j] for j in range(n_bars)])):
    for k, patch in enumerate(patch_group):
        new_y = i * group_gap + (k - (n_bars - 1) / 2) * bar_width
        patch.set_y(new_y - bar_width / 2)
        patch.set_height(bar_width)

ax.set_yticks([i * group_gap for i in range(n_groups)])
ax.set_yticklabels(ALGEBRA_ORDER, fontsize=16)
ax.tick_params(axis='x', labelsize=17)
ax.legend(fontsize=15, loc='upper right')
ax.set_ylim(-0.5, (n_groups - 1) * group_gap + 0.5)
ax.invert_yaxis()
ax.set_xscale('log', base=10)
ax.set_xlabel('Relative Frequency of Algebra Labels\nwith respect to Training Data', fontsize=18)
ax.set_ylabel('Algebra Label', fontsize=18)
ax.axvline(x=1, color='black', linestyle='--', linewidth=1)
plt.tight_layout()
plt.savefig('algebra_freq_all_range_rel.pdf')
plt.show()

In [ ]:
# Fig 4 RIGHT — relative central charge denominator frequency (horizontal)
y      = np.arange(len(all_denoms))
height = 0.25

fig, ax = plt.subplots(figsize=(6, 8))
for i, (key, color) in enumerate(zip(REL_KEYS, REL_COLORS)):
    ax.barh(y + (i - 1) * height, cc_rel[key], height, label=key, color=color)

ax.set_yticks(y)
ax.set_yticklabels([str(d) for d in all_denoms], fontsize=17)
ax.set_ylim(y.max() + 0.42, y.min() - 0.42)
ax.set_ylabel('Denominator of Central Charge', fontsize=20)
ax.set_xlabel('Relative Frequency of Central Charges\nwith respect to Training Data', fontsize=18)
ax.axvline(x=1, color='black', linestyle='--', linewidth=1)
ax.tick_params(axis='both', labelsize=18)
ax.legend(fontsize=15, loc='upper right')
ax.set_xscale('log', base=10)
plt.tight_layout()
plt.savefig('cc_freq_all_range_rel.pdf')
plt.show()